In [ ]:
#Cell that you requested :)
import logging
from pathlib import Path

import h5py
import hist
import matplotlib.pyplot as plt
import mplhep as hep
import numpy as np
from hist.intervals import clopper_pearson_interval

hep.style.use(hep.style.ROOT)
logging.basicConfig(level=logging.INFO)

out_dir = Path("/Users/juju/SpaTop/local_data/validation_output")

# The two files copied down from /data/spatop/tttt_15M/h5/ on the cluster.
# These are already-processed training datasets (built by convert_to_h5.py),
# not raw Delphes ROOT ntuples -- the top<->jet matching is precomputed and
# stored directly as TARGETS/<category><top_slot>/{MASK,pt,...}, with up to
# 4 top slots (t1..t4), one per reconstruction category: FR (fully-resolved),
# FB (fully-boosted), SRqq / SRbq (semi-resolved).
FILES = {
    "testing": "/Users/juju/SpaTop/local_data/tttt_15M/h5/tttt_testing.h5",
    "training": "/Users/juju/SpaTop/local_data/tttt_15M/h5/tttt_training.h5",
}
N_TOPS = 4
CATEGORIES = ["FR", "FB", "SRqq", "SRbq"]
# Fixed per-category colors so the top (distribution) and bottom (efficiency)
# panels are visually consistent -- hep.histplot's auto-cycling otherwise
# restarts independently per axis and the two panels end up mismatched.
COLORS = {"All tops": "black", "FR": "C0", "FB": "C1", "SRqq": "C2", "SRbq": "C3", "any": "C4"}


def load_category_pt_mask(f, category, n_tops):
    """Concatenate pt & MASK across all top slots (t1..n_tops) for one category."""
    pts, masks = [], []
    for i in range(1, n_tops + 1):
        group = f"TARGETS/{category}t{i}"
        pts.append(f[f"{group}/pt"][:])
        masks.append(f[f"{group}/MASK"][:])
    return np.concatenate(pts), np.concatenate(masks)


for tag, path in FILES.items():
    print(f"=== {tag}: {path} ===")
    with h5py.File(path, "r") as f:
        n_events = f["TARGETS/FRt1/MASK"].shape[0]
        print(f"n_events = {n_events}")

        # "all tops" pt distribution: top{i}_pt is identical across categories,
        # so any one category's pt field represents the full set of generator
        # tops (padded slots are filled with -1, so pt > 0 selects real tops).
        all_pt = np.concatenate([f[f"TARGETS/FRt{i}/pt"][:] for i in range(1, N_TOPS + 1)])
        all_pt = all_pt[all_pt > 0]

        all_top_hist = hist.Hist.new.Reg(10, 0, 1000, name=r"top $p_T$ [GeV]").Double()
        all_top_hist.fill(all_pt)

        cat_hists = {}
        merged_matched_any = None  # union across categories, aligned per (event, top slot)
        for category in CATEGORIES:
            try:
                pt, mask = load_category_pt_mask(f, category, N_TOPS)
            except KeyError as e:
                # tttt_testing.h5 has real internal corruption in its
                # SRqqt*/SRbqt* groups (confirmed: same fields read fine in
                # tttt_training.h5, and low-level h5py group ops on this file
                # fail with "bad symbol table node signature") -- skip rather
                # than crash the whole cell, but surface it clearly.
                print(f"  SKIPPING category={category} for {tag}: unreadable ({e}) -- likely file corruption")
                continue

            matched_pt = pt[(pt > 0) & mask]
            h = hist.Hist.new.Reg(10, 0, 1000, name=r"top $p_T$ [GeV]").Double()
            h.fill(matched_pt)
            cat_hists[category] = h

            valid = pt > 0
            if merged_matched_any is None:
                merged_matched_any = np.zeros_like(valid, dtype=bool)
            merged_matched_any = merged_matched_any | (valid & mask)

        # merged: matched by *any* available category (only counts categories
        # that were actually readable for this file)
        merged_pt = np.concatenate([f[f"TARGETS/FRt{i}/pt"][:] for i in range(1, N_TOPS + 1)])
        merged_valid = merged_pt > 0
        merged_matched_pt = (
            merged_pt[merged_valid][merged_matched_any[merged_valid]] if merged_matched_any is not None else np.array([])
        )
        merged_hist = hist.Hist.new.Reg(10, 0, 1000, name=r"top $p_T$ [GeV]").Double()
        merged_hist.fill(merged_matched_pt)

        # --- plot: distribution + efficiency ---
        fig, axs = plt.subplots(2, 1, height_ratios=[2, 1])
        hep.histplot(all_top_hist, label="All tops", ax=axs[0], color=COLORS["All tops"])
        for category, h in cat_hists.items():
            hep.histplot(h, label=f"matched ({category})", ax=axs[0], color=COLORS[category])
        hep.histplot(merged_hist, label="matched (any category)", ax=axs[0], color=COLORS["any"])
        axs[0].set_ylabel("Top quarks")
        axs[0].set_xlim(0, 1000)
        axs[0].set_ylim(1e-1, 1e7)
        axs[0].semilogy()
        axs[0].legend(loc="upper right", fontsize=10)
        axs[0].set_title(f"{tag} ({Path(path).name})")

        for category, h in cat_hists.items():
            ratio = h / all_top_hist
            uncert = np.abs(clopper_pearson_interval(num=h.values(), denom=all_top_hist.values()) - ratio)
            hep.histplot(ratio, yerr=uncert, label=category, ax=axs[1], color=COLORS[category])
        merged_ratio = merged_hist / all_top_hist
        merged_uncert = np.abs(clopper_pearson_interval(num=merged_hist.values(), denom=all_top_hist.values()) - merged_ratio)
        hep.histplot(merged_ratio, yerr=merged_uncert, label="any", ax=axs[1], color=COLORS["any"])
        axs[1].set_ylabel("Efficiency")
        axs[1].set_xlim(0, 1000)
        axs[1].set_ylim(-0.05, 1.05)
        axs[1].legend(loc="upper left", fontsize=8, ncol=5)
        plt.tight_layout()
        fig.savefig(out_dir / f"h5_top_pt_{tag}.png")
        fig.savefig(out_dir / f"h5_top_pt_{tag}.pdf")
        plt.show()

        # jet / fatjet multiplicity, from the precomputed INPUTS masks
        n_jets_arr = f["INPUTS/Jets/MASK"][:].sum(axis=1)
        n_fjets_arr = f["INPUTS/BoostedJets/MASK"][:].sum(axis=1)

        plt.figure()
        n_jets_h = hist.Hist.new.Reg(17, -0.5, 16.5, name="AK5 Jets").Double()
        n_jets_h.fill(n_jets_arr)
        hep.histplot(n_jets_h)
        plt.ylabel("Events")
        plt.xlabel("AK5 Jets")
        plt.semilogy()
        plt.title(tag)
        plt.tight_layout()
        plt.savefig(out_dir / f"h5_n_jets_{tag}.png")
        plt.savefig(out_dir / f"h5_n_jets_{tag}.pdf")
        plt.show()

        plt.figure()
        n_fjets_h = hist.Hist.new.Reg(6, -0.5, 5.5, name="AK8 Jets").Double()
        n_fjets_h.fill(n_fjets_arr)
        hep.histplot(n_fjets_h)
        plt.ylabel("Events")
        plt.xlabel("AK8 Jets")
        plt.semilogy()
        plt.title(tag)
        plt.tight_layout()
        plt.savefig(out_dir / f"h5_n_fjets_{tag}.png")
        plt.savefig(out_dir / f"h5_n_fjets_{tag}.pdf")
        plt.show()


In [ ]:
# ============================================================
# Dataset kinematic plots (tt, tttt) + target jet-overlap check
# ------------------------------------------------------------
# Plain kinematic distributions (no matching/efficiency math -- that's
# already covered by the validation cells above via calc_pur_eff):
#   1. top pt for ALL events, and per matched top class, for the FULL
#      dataset (training+testing combined) -- for both tt and tttt.
#   2. top pt per matched top class, for training and testing separately.
#   3. AK4/AK8 jet-multiplicity distributions (plain event-kinematic).
#   4. a check on whether jets assigned to one top slot (e.g. FRt1) ever
#      overlap with jets assigned to another top slot (FRt2, ...) in the
#      same event/category -- nothing in the target construction enforces
#      that they don't.
# ============================================================
import h5py
import hist
import matplotlib.pyplot as plt
import mplhep as hep
import numpy as np
from pathlib import Path

hep.style.use(hep.style.ROOT)

out_dir = Path("/Users/juju/SpaTop/local_data/validation_output")

CATEGORIES = ["FR", "FB", "SRqq", "SRbq"]
COLORS = {"All tops": "black", "FR": "C0", "FB": "C1", "SRqq": "C2", "SRbq": "C3"}

# jet-index fields per category, tagged by which INPUTS collection they index
# into (AK4 = INPUTS/Jets, AK8 = INPUTS/BoostedJets). Needed for the overlap
# check: index i in Jets and index i in BoostedJets are different physical
# objects, so "shared jet" can only be compared within the same collection.
CATEGORY_INDEX_FIELDS = {
    "FR": {"AK4": ["b", "q1", "q2"], "AK8": []},
    "FB": {"AK4": [], "AK8": ["bqq"]},
    "SRqq": {"AK4": ["b"], "AK8": ["qq"]},
    "SRbq": {"AK4": ["q"], "AK8": ["bq"]},
}

DATASETS = {
    "tt": {
        "n_tops": 2,
        "files": {
            "training": "/Users/juju/SpaTop/v8/tt_hadronic_fixed_train.h5",
            "testing": "/Users/juju/SpaTop/v8/tt_hadronic_fixed_test.h5",
        },
    },
    "tttt": {
        "n_tops": 4,
        "files": {
            # tttt_testing.h5 has real internal corruption in its SRqqt*/SRbqt*
            # groups (low-level h5py group ops fail with "bad symbol table
            # node signature") -- handled below by skipping just those parts.
            "training": "/Users/juju/SpaTop/local_data/tttt_15M/h5/tttt_training.h5",
            "testing": "/Users/juju/SpaTop/local_data/tttt_15M/h5/tttt_testing.h5",
        },
    },
}


def read_field(paths, dset):
    """Read + concatenate one h5 dataset path across one or more files.
    Returns None if `dset` is missing/unreadable in any of the files."""
    out = []
    for p in paths:
        with h5py.File(p, "r") as f:
            try:
                if dset not in f:
                    return None
                out.append(f[dset][:])
            except RuntimeError:
                return None
    return np.concatenate(out)


def load_top_pt(paths, n_tops):
    """All generator-top pt across top slots t1..n_tops (padded slots pt<=0 dropped)."""
    pt = np.concatenate([read_field(paths, f"TARGETS/FRt{i}/pt") for i in range(1, n_tops + 1)])
    return pt[pt > 0]


def load_category_matched_pt(paths, category, n_tops):
    """Concatenate matched-top pt across top slots t1..n_tops for one category."""
    pts, masks = [], []
    for i in range(1, n_tops + 1):
        pt = read_field(paths, f"TARGETS/{category}t{i}/pt")
        mask = read_field(paths, f"TARGETS/{category}t{i}/MASK")
        if pt is None or mask is None:
            return None
        pts.append(pt)
        masks.append(mask)
    pt, mask = np.concatenate(pts), np.concatenate(masks)
    return pt[(pt > 0) & mask]


def plot_top_pt(dataset_name, tag, paths, n_tops, include_all=True):
    """Plain top-pt distribution: all tops (optional) + one histogram per matched category."""
    fig, ax = plt.subplots()

    if include_all:
        all_top_hist = hist.Hist.new.Reg(10, 0, 1000, name=r"top $p_T$ [GeV]").Double()
        all_top_hist.fill(load_top_pt(paths, n_tops))
        hep.histplot(all_top_hist, label="All tops", ax=ax, color=COLORS["All tops"])

    for category in CATEGORIES:
        matched_pt = load_category_matched_pt(paths, category, n_tops)
        if matched_pt is None:
            print(f"  SKIPPING category={category} for {dataset_name}/{tag}: unreadable -- likely file corruption")
            continue
        h = hist.Hist.new.Reg(10, 0, 1000, name=r"top $p_T$ [GeV]").Double()
        h.fill(matched_pt)
        hep.histplot(h, label=f"matched ({category})", ax=ax, color=COLORS[category])

    ax.set_ylabel("Top quarks")
    ax.set_xlim(0, 1000)
    ax.set_ylim(1e-1, 1e7)
    ax.semilogy()
    ax.legend(loc="upper right", fontsize=10)
    ax.set_title(f"{dataset_name} -- {tag}")
    plt.tight_layout()
    fig.savefig(out_dir / f"h5_top_pt_{dataset_name}_{tag}.png")
    fig.savefig(out_dir / f"h5_top_pt_{dataset_name}_{tag}.pdf")
    plt.show()


def plot_jet_multiplicity(dataset_name, tag, paths):
    """AK4/AK8 jet-multiplicity distributions, from the precomputed INPUTS masks.
    Bin count is sized to each dataset's actual max slot count (tt: 10 AK4 / 3 AK8,
    tttt: 16 AK4 / 5 AK8) rather than assuming a fixed size."""
    for label, dset, tagname in [
        ("AK5 Jets", "INPUTS/Jets/MASK", "jets"),
        ("AK8 Jets", "INPUTS/BoostedJets/MASK", "fjets"),
    ]:
        n_arr = read_field(paths, dset).sum(axis=1)
        n_max = n_arr.max()
        plt.figure()
        h = hist.Hist.new.Reg(n_max + 2, -0.5, n_max + 1.5, name=label).Double()
        h.fill(n_arr)
        hep.histplot(h)
        plt.ylabel("Events")
        plt.xlabel(label)
        plt.semilogy()
        plt.title(f"{dataset_name} -- {tag}")
        plt.tight_layout()
        plt.savefig(out_dir / f"h5_n_{tagname}_{dataset_name}_{tag}.png")
        plt.savefig(out_dir / f"h5_n_{tagname}_{dataset_name}_{tag}.pdf")
        plt.show()


def check_target_overlap(dataset_name, tag, paths, n_tops):
    """
    For each category, check whether two *simultaneously matched* top slots
    in the same event share an assigned jet in the same INPUTS collection.
    Nothing in the target construction requires the jets picked for one top
    to be disjoint from the jets picked for another top -- this quantifies
    how often that actually happens.
    """
    print(f"--- target jet-overlap check: {dataset_name}/{tag} ---")
    for category in CATEGORIES:
        fields = CATEGORY_INDEX_FIELDS[category]
        masks, idxs = [], {"AK4": [], "AK8": []}
        readable = True
        for i in range(1, n_tops + 1):
            mask = read_field(paths, f"TARGETS/{category}t{i}/MASK")
            if mask is None:
                readable = False
                break
            masks.append(mask)
            for coll, cols in fields.items():
                idxs[coll].append(
                    np.stack([read_field(paths, f"TARGETS/{category}t{i}/{c}") for c in cols], axis=-1)
                    if cols
                    else None
                )
        if not readable:
            print(f"  {category}: SKIPPED (unreadable -- likely file corruption)")
            continue

        n_events = len(masks[0])
        overlap_any = np.zeros(n_events, dtype=bool)
        both_matched_any = np.zeros(n_events, dtype=bool)
        for i in range(n_tops):
            for j in range(i + 1, n_tops):
                both_matched = masks[i] & masks[j]
                both_matched_any |= both_matched
                pair_overlap = np.zeros(n_events, dtype=bool)
                for coll in ("AK4", "AK8"):
                    idx_i, idx_j = idxs[coll][i], idxs[coll][j]
                    if idx_i is None or idx_j is None:
                        continue
                    pair_overlap |= (idx_i[:, :, None] == idx_j[:, None, :]).any(axis=(1, 2))
                overlap_any |= pair_overlap & both_matched

        n_multi = int(both_matched_any.sum())
        n_bad = int(overlap_any.sum())
        frac = n_bad / n_multi if n_multi else float("nan")
        print(f"  {category}: {n_bad}/{n_multi} events with >=2 matched tops share a target jet ({frac:.2%})")


for dataset_name, cfg in DATASETS.items():
    n_tops = cfg["n_tops"]
    files = cfg["files"]

    print(f"=== {dataset_name} ===")

    # full dataset (training + testing combined): all events + per matched top class
    full_paths = list(files.values())
    plot_top_pt(dataset_name, "full", full_paths, n_tops, include_all=True)
    plot_jet_multiplicity(dataset_name, "full", full_paths)
    check_target_overlap(dataset_name, "full", full_paths, n_tops)

    # per matched top class pt, training vs. testing separately
    for tag, path in files.items():
        plot_top_pt(dataset_name, tag, [path], n_tops, include_all=False)
        check_target_overlap(dataset_name, tag, [path], n_tops)
